In [0]:
from pyspark.sql import functions as F, Window as W

BRONZE = "workspace.climate_bronze"
SILVER = "workspace.climate_silver"


def latest_only(df, keys):
    """Keep only the most recently ingested row per key (handles re-delivered files)."""
    w = W.partitionBy(*keys).orderBy(F.col("_ingested_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")


def save(df, table):
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{SILVER}.{table}"))
    print(f"{table}: {spark.table(f'{SILVER}.{table}').count():,} rows")

In [0]:
districts = latest_only(spark.table(f"{BRONZE}.districts"), ["district_id"]).select(
    "district_id",
    "district_name",
    "county_name",
    F.col("lat").cast("double").alias("lat"),
    F.col("lon").cast("double").alias("lon"),
    F.col("area_km2").cast("double").alias("area_km2"),
)
save(districts, "districts")

In [0]:
def num(c):
    """Cast to number and turn NASA's -999 'missing' marker into a real null."""
    x = F.col(c).cast("double")
    return F.when(x <= -999, None).otherwise(x)


typed = spark.table(f"{BRONZE}.weather_daily").select(
    "district_id",
    F.expr("date_add(make_date(CAST(YEAR AS INT), 1, 1), CAST(DOY AS INT) - 1)").alias("date"),
    num("PRECTOTCORR").alias("precip_mm"),
    num("T2M_MAX").alias("tmax_c"),
    num("T2M_MIN").alias("tmin_c"),
    num("GWETROOT").alias("root_soil_wetness"),
    "_source_file",
    "_ingested_at",
)

weather = latest_only(typed, ["district_id", "date"])

# referential integrity: every row must belong to a known district
known = spark.table(f"{SILVER}.districts").select("district_id").withColumn("_known", F.lit(True))
weather = weather.join(known, "district_id", "left")

rules = {
    "unknown_district":   F.col("_known").isNull(),
    "missing_date":       F.col("date").isNull(),
    "rain_out_of_range":  ~F.col("precip_mm").between(0, 500),
    "temp_out_of_range":  ~F.col("tmax_c").between(-10, 55),
    "tmax_below_tmin":    F.col("tmax_c") < F.col("tmin_c"),
    "soil_wetness_invalid": ~F.col("root_soil_wetness").between(0, 1),
}

# list every rule a row breaks (nulls don't count as failures; they're handled later)
checked = weather.withColumn(
    "failed_rules",
    F.concat_ws(", ", *[F.when(cond, F.lit(name)) for name, cond in rules.items()]),
).drop("_known")

save(checked.filter("failed_rules = ''").drop("failed_rules"), "weather_daily")
save(checked.filter("failed_rules != ''"), "weather_quarantine")

In [0]:
thickness = (F.when(F.col("depth") == "0-5cm", 5)
              .when(F.col("depth") == "5-15cm", 10)
              .when(F.col("depth") == "15-30cm", 15))

soil_long = (latest_only(spark.table(f"{BRONZE}.soil_raw"), ["district_id", "property", "depth"])
             .withColumn("value", F.col("value_mapped").cast("double") / 10)  # -> clay %, sand %, SOC g/kg
             .withColumn("thickness_cm", thickness))

# depth-weighted average over 0-30 cm, ignoring missing layers
soil_wide = (soil_long.groupBy("district_id")
             .pivot("property", ["clay", "sand", "soc"])
             .agg(F.sum(F.col("value") * F.col("thickness_cm"))
                  / F.sum(F.when(F.col("value").isNotNull(), F.col("thickness_cm"))))
             .withColumnRenamed("clay", "clay_pct")
             .withColumnRenamed("sand", "sand_pct")
             .withColumnRenamed("soc", "soc_gkg"))

cols = ["clay_pct", "sand_pct", "soc_gkg"]
soil = spark.table(f"{SILVER}.districts").select("district_id", "county_name").join(soil_wide, "district_id", "left")

county_med = soil.groupBy("county_name").agg(*[F.median(c).alias(f"{c}_county") for c in cols])
national_med = soil.agg(*[F.median(c).alias(f"{c}_national") for c in cols])

soil = (soil.join(county_med, "county_name", "left")
            .crossJoin(national_med)
            .withColumn("soil_imputed", F.col("soc_gkg").isNull() | F.col("clay_pct").isNull()))

for c in cols:
    soil = soil.withColumn(c, F.round(F.coalesce(c, f"{c}_county", f"{c}_national"), 1))

save(soil.select("district_id", *cols, "soil_imputed"), "district_soil")

In [0]:
comments = {
    "districts": "Kenyan sub-counties (geoBoundaries ADM2) with representative point and county",
    "weather_daily": "Validated daily weather per sub-county from NASA POWER, 1985-2025",
    "weather_quarantine": "Weather rows that failed validation, with failed_rules listing why",
    "district_soil": "0-30cm depth-weighted soil properties from SoilGrids; soil_imputed marks county/national median fills",
}
for table, text in comments.items():
    spark.sql(f"COMMENT ON TABLE {SILVER}.{table} IS '{text}'")
print("comments added")

In [0]:
w = spark.table(f"{SILVER}.weather_daily")
q = spark.table(f"{SILVER}.weather_quarantine")
s = spark.table(f"{SILVER}.district_soil")

print(f"weather rows:      {w.count():,}")
print(f"quarantined rows:  {q.count():,}")
print(f"date range:        {w.agg(F.min('date'), F.max('date')).first()}")
print(f"soil rows:         {s.count()}   imputed: {s.filter('soil_imputed').count()}")

if q.count() > 0:
    display(q.groupBy("failed_rules").count().orderBy(F.desc("count")))

display(s.filter("soil_imputed").join(spark.table(f"{SILVER}.districts"), "district_id")
         .select("county_name", "district_name", "clay_pct", "sand_pct", "soc_gkg"))

In [0]:
display(
    spark.table(f"{SILVER}.weather_quarantine")
    .join(spark.table(f"{SILVER}.districts"), "district_id")
    .select("county_name", "district_name", "date",
            "precip_mm", "tmax_c", "tmin_c", "root_soil_wetness", "failed_rules")
    .orderBy("date")
)